# Day 1 — Machine Learning with Scikit-learn
**ComplianceGPT Lab · REU 2026 · Monday June 29**

The year is 1990. You need to classify whether a hospital disclosure complies with HIPAA.
You have no LLMs. You have no neural networks. You have: **features you write by hand** and a learning algorithm.

By the end of this notebook you'll have trained a classifier that gets ~70% accuracy — and hit a wall that motivates everything that comes next.

**No GPU needed. Runs on your laptop.**


---
## Part 1 — The Feature Engineering Problem

Before ML can learn anything, someone has to decide *what features to extract from the text*.
For HIPAA compliance, the relevant features are things like:
- Does the scenario mention a court order?
- Is the recipient a law enforcement officer?
- Is there patient authorization?

We'll hand-engineer these features, then let a model learn which ones matter.

In [ ]:
# ── Run this cell first ──────────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report

print('All imports OK')

In [ ]:
# ── Simulated HIPAA scenario dataset (80 cases) ─────────────────────────
# Each row = one court case scenario, pre-labeled PERMITTED or DENIED
# Features are boolean: did a human annotator mark this scenario as having X?
#
# In real life YOU extract these features from raw text using LLMs (later this week)
# Here we're skipping the extraction to focus on the ML part

np.random.seed(42)
n = 80

# Feature definitions
features = [
    'has_court_order',           # did a judge issue a formal order?
    'receiver_is_law_enforcement', # is the recipient a cop/FBI/DEA?
    'is_required_by_law',         # does a statute compel disclosure?
    'has_patient_auth',           # did the patient sign an authorization?
    'has_treatment_relationship', # is this a treating provider?
    'purpose_is_treatment',       # is the purpose patient treatment?
    'purpose_is_research',        # is this a research disclosure?
    'has_lawful_process',         # subpoena / administrative demand?
    'receiver_is_researcher',     # is the recipient a researcher?
    'phi_is_mental_health',       # extra-sensitive category?
]

# Generate plausible data:
# PERMITTED requires at least one enabling condition
X_raw = np.random.randint(0, 2, size=(n, len(features)))
# Simplified rule: PERMITTED if court_order OR patient_auth OR (treatment_relationship AND treatment_purpose)
y = np.where(
    (X_raw[:,0] == 1) |
    (X_raw[:,3] == 1) |
    ((X_raw[:,4] == 1) & (X_raw[:,5] == 1)),
    'PERMITTED', 'DENIED'
)

df = pd.DataFrame(X_raw, columns=features)
df['label'] = y

print(f'Dataset: {n} scenarios')
print(df['label'].value_counts())
df.head(5)

## Part 2 — Train Three Classifiers

We'll try three classic ML algorithms and compare them.
All from scikit-learn — no deep learning yet.

| Model | How it works |
|-------|-------------|
| Logistic Regression | Learns a weight for each feature — fast, interpretable |
| Decision Tree | Learns a series of if/else rules from data |
| Random Forest | 100 decision trees vote — usually more accurate |

In [ ]:
X = df[features].values
y_labels = df['label'].values

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
}

print('Model              │ Accuracy (5-fold CV)')
print('────────────────────┼────────────────────')
for name, model in models.items():
    scores = cross_val_score(model, X, y_labels, cv=5, scoring='accuracy')
    print(f'{name:<20} │ {scores.mean():.1%} ± {scores.std():.1%}')

## Part 3 — What Did the Model Learn?

The best part about logistic regression and decision trees: you can *read* what they learned.

In [ ]:
# ── Logistic Regression weights ─────────────────────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=42).fit(X, y_labels)
weights = pd.Series(lr.coef_[0], index=features).sort_values(ascending=False)

print('Logistic Regression — feature weights (higher = more PERMITTED):')
print(weights.to_string())

print()

# ── Decision Tree rules ──────────────────────────────────────────────────
dt = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X, y_labels)
print('Decision Tree rules (first 3 levels):')
print(export_text(dt, feature_names=features, max_depth=3))

## Part 4 — Hit the Wall

Our model learned from boolean features. But in real life, **you don't have those features yet** — you have raw text:

> *"The hospital received what the officer described as a possible court order and disclosed the patient's records."*

Does this scenario have `has_court_order = True` or `False`?

A keyword search for 'court order' returns True — but there's no confirmed court order. 
A human reads this and says False. The LLM has to read it and decide.

**This is the extraction problem.** The sklearn classifier is correct IF the features are correct.
The hard part — which you'll work on this week — is extracting those features from text.

In [ ]:
# ── YOUR CODE ────────────────────────────────────────────────────────────
# Train a Random Forest and print its feature importances
# Which feature does the forest consider most important? Does it match your intuition?

rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X, y_labels)
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)

# YOUR CODE: print importances and answer the reflection below
print(importances)

**Reflection (fill in below):**

1. Which feature had the highest importance in your Random Forest? Does that make legal sense?

2. Our dataset used hand-crafted boolean features. What would break if you tried to use raw text instead of these features?

3. Our classifier gets ~90% on this simulated data. But in the GoldCoin benchmark, even LLMs only get 94%. Why might real data be harder than this simulation?

*Your answers here:*


---
## Submit

When you're done: post this notebook in Slack `#deliverables` with your name in the filename.
No due date pressure — submit when complete.

**Next:** Day 2 (`day2_nlp.ipynb`) — we drop the hand-crafted features and work directly with text.